In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to the MIMIC-IV dataset
mimic_path = "/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0"
output_path = '/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/'

Mounted at /content/drive


## Python conversion of dopamine-dose.sql for MIMIC-IV

In [3]:
# Python conversion of dopamine-dose.sql for MIMIC-IV

import pandas as pd
import numpy as np
from datetime import timedelta

def create_dopamine_dose(mimic_path, output_path, weightdurations=None):
    """
    Python conversion of dopamine-dose.sql for MIMIC-IV
    Creates a dataframe with dopamine dose and duration information

    Parameters:
    mimic_path (str): Path to MIMIC-IV data
    output_path (str): Path to save the output dataframe
    weightdurations (pd.DataFrame, optional): Weightdurations dataframe

    Returns:
    pd.DataFrame: Dopamine dose dataframe
    """
    print("Creating dopamine dose dataframe...")

    # Load inputevents table (combined table in MIMIC-IV)
    print("Loading inputevents table...")
    inputevents = pd.read_csv(
        f"{mimic_path}/icu/inputevents.csv.gz",
        usecols=['stay_id', 'starttime', 'endtime', 'itemid', 'rate', 'amount', 'orderid', 'statusdescription'],
        parse_dates=['starttime', 'endtime']
    )

    # Rename stay_id to icustay_id for consistency with original SQL
    inputevents.rename(columns={'stay_id': 'icustay_id', 'orderid': 'linkorderid'}, inplace=True)

    # Filter for dopamine
    # Note: MIMIC-IV itemids may differ from MIMIC-III
    # You'll need to update these itemids based on MIMIC-IV dictionary
    dopamine_itemid = 221662  # Update this with the correct MIMIC-IV itemid for dopamine

    # Filter for valid orders
    dopamine_mv = inputevents[
        (inputevents['itemid'] == dopamine_itemid) &
        (inputevents['statusdescription'] != 'Rewritten')
    ].copy()

    # Group by icustay_id and linkorderid
    dopamine_mv_grouped = dopamine_mv.groupby(['icustay_id', 'linkorderid']).agg({
        'rate': 'max',
        'amount': 'sum',
        'starttime': 'min',
        'endtime': 'max'
    }).reset_index()

    dopamine_mv_grouped.rename(columns={
        'rate': 'vaso_rate',
        'amount': 'vaso_amount'
    }, inplace=True)

    # Sort the final dataframe
    dopamine_dose = dopamine_mv_grouped.sort_values(['icustay_id', 'starttime'])

    # Save to CSV
    dopamine_dose.to_csv(f"{output_path}/dopamine_dose.csv", index=False)
    print(f"Dopamine dose saved to {output_path}/dopamine_dose.csv")

    return dopamine_dose

# Call the function directly with the defined paths
create_dopamine_dose(mimic_path, output_path)

Creating dopamine dose dataframe...
Loading inputevents table...
Dopamine dose saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data//dopamine_dose.csv


,icustay_id,linkorderid,vaso_rate,vaso_amount,starttime,endtime
0,30000484,816461,5.003784,165.209934,2136-01-15 09:40:00,2136-01-15 17:42:00
2,30000484,2425757,2.501892,34.275920,2136-01-15 17:42:00,2136-01-15 21:02:00
1,30000484,1892112,1.557223,64.001852,2136-01-15 21:02:00,2136-01-16 07:02:00
3,30001446,726899,5.004316,19.701491,2186-04-12 05:36:00,2186-04-12 06:09:00
4,30001446,3555543,2.502158,23.582088,2186-04-12 06:09:00,2186-04-12 07:28:00
...,...,...,...,...,...,...
18081,39993968,6391069,15.430618,146.397982,2170-06-20 00:40:00,2170-06-20 03:25:00
18082,39993968,7417811,15.065367,144.665185,2170-06-20 03:25:00,2170-06-20 06:12:00
18079,39993968,2607961,10.049626,238.075642,2170-06-20 06:12:00,2170-06-20 13:04:00
18084,39996783,2670268,10.004002,5.462185,2126-06-29 06:10:00,2126-06-29 06:23:00


In [ ]:
# List the contents of the mimic_path directory to verify the file path
import os
print(f"Contents of {mimic_path}:")
try:
    for item in os.listdir(mimic_path):
        print(item)
except FileNotFoundError:
    print("Error: The specified mimic_path was not found.")

Contents of /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0:
CHANGELOG.txt
LICENSE.txt
hosp
icu
index.html
ventilation_data


## Python conversion of epinephrine_dose.sql for MIMIC-IV

In [4]:
# Python conversion of epinephrine_dose.sql for MIMIC-IV

import pandas as pd
import numpy as np
from datetime import timedelta

def create_epinephrine_dose(mimic_path, output_path, weightdurations=None):
    """
    Python conversion of epinephrine_dose.sql for MIMIC-IV
    Creates a dataframe with epinephrine dose and duration information

    Parameters:
        mimic_path (str): Path to MIMIC-IV data
        output_path (str): Path to save the output dataframe
        weightdurations (pd.DataFrame, optional): Weightdurations dataframe

    Returns:
        pd.DataFrame: Epinephrine dose dataframe
    """
    print("Creating epinephrine dose dataframe...")

    # Load inputevents table (combined table in MIMIC-IV)
    print("Loading inputevents table...")
    try:
        inputevents = pd.read_csv(
            f"{mimic_path}/icu/inputevents.csv.gz",
            usecols=['stay_id', 'starttime', 'endtime', 'itemid', 'rate', 'amount', 'orderid', 'statusdescription'],
            parse_dates=['starttime', 'endtime']
        )
        print(f"Successfully loaded inputevents. Shape: {inputevents.shape}, Type: {type(inputevents)}")
    except FileNotFoundError:
        print(f"Error: inputevents.csv.gz not found at {mimic_path}/icu/")
        return None # Exit the function if the file is not found
    except Exception as e:
        print(f"An error occurred while loading inputevents: {e}")
        return None # Exit the function if there's an error during loading


    # Rename stay_id to icustay_id for consistency with original SQL
    inputevents.rename(
        columns={'stay_id': 'icustay_id', 'orderid': 'linkorderid'},
        inplace=True
    )

    # Load weightdurations if not provided
    if weightdurations is None:
        try:
            weightdurations = pd.read_csv(
                f"{output_path}/weightdurations.csv",
                parse_dates=['starttime', 'endtime']
            )
        except:
          print("Weightdurations file not found. Weight-based calculations may be inaccurate.")
          weightdurations = None

    # Filter for epinephrine
    # Note: MIMIC-IV itemids may differ from MIMIC-III
    # You'll need to update these itemids based on MIMIC-IV dictionary
    epinephrine_itemid = 221289  # Update this with the correct MIMIC-IV itemid for epinephrine

    # Check if inputevents is defined before filtering
    if 'inputevents' in locals() and isinstance(inputevents, pd.DataFrame):
        print("inputevents DataFrame is defined and ready for filtering.")
        # Filter for valid orders
        epinephrine_mv = inputevents[
            (inputevents['itemid'] == epinephrine_itemid) &
            (inputevents['statusdescription'] != 'Rewritten')
        ].copy()

        # If weightdurations is available, join to get weight-adjusted rates
        if weightdurations is not None:
            # For each infusion, find the appropriate weight
            # This is a simplification of the SQL logic
            def find_weight(row):
                matching_weights = weightdurations[
                    (weightdurations['icustay_id'] == row['icustay_id']) &
                    (row['starttime'] >= weightdurations['starttime']) &
                    (row['starttime'] <= weightdurations['endtime'])
                ]
                if not matching_weights.empty:
                    return matching_weights.iloc[0]['weight']
                return 80.0  # Default weight if no match

            epinephrine_mv['weight'] = epinephrine_mv.apply(find_weight, axis=1)

            # Adjust rate based on weight where needed
            # This is a simplification of the SQL logic
            epinephrine_mv['vaso_rate'] = epinephrine_mv['rate']  # Already in mcgkgmin in MIMIC-IV
        else:
            epinephrine_mv['vaso_rate'] = epinephrine_mv['rate']

        # Group by icustay_id and linkorderid
        epinephrine_mv_grouped = epinephrine_mv.groupby(['icustay_id', 'linkorderid']).agg({
            'vaso_rate': 'max',
            'amount': 'sum',
            'starttime': 'min',
            'endtime': 'max'
        }).reset_index()

        epinephrine_mv_grouped.rename(columns={
            'amount': 'vaso_amount'
        }, inplace=True)

        # Sort the final dataframe
        epinephrine_dose = epinephrine_mv_grouped.sort_values(['icustay_id', 'starttime'])

        # Save to CSV
        epinephrine_dose.to_csv(f"{output_path}/epinephrine_dose.csv", index=False)
        print(f"Epinephrine dose saved to {output_path}/epinephrine_dose.csv")

        return epinephrine_dose
    else:
        print("inputevents DataFrame was not successfully created.")
        return None


# Call the function directly with the defined paths
print(f"Using mimic_path: {mimic_path}")
print(f"Using output_path: {output_path}")
create_epinephrine_dose(mimic_path, output_path)

Using mimic_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0
Using output_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/
Creating epinephrine dose dataframe...
Loading inputevents table...
Successfully loaded inputevents. Shape: (10953713, 8), Type: <class 'pandas.core.frame.DataFrame'>
inputevents DataFrame is defined and ready for filtering.
Epinephrine dose saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data//epinephrine_dose.csv


,icustay_id,linkorderid,vaso_rate,vaso_amount,starttime,endtime
15,30003749,6775694,0.502343,7.910400,2180-06-06 16:30:00,2180-06-06 19:24:00
16,30003749,6937933,0.502343,7.910400,2180-06-06 19:24:00,2180-06-06 22:18:00
7,30003749,3053924,0.502302,7.955201,2180-06-06 22:18:00,2180-06-07 01:13:00
9,30003749,3238238,0.502343,7.910400,2180-06-07 01:13:00,2180-06-07 04:07:00
14,30003749,6468412,0.502343,7.910400,2180-06-07 04:07:00,2180-06-07 07:01:00
...,...,...,...,...,...,...
31474,39995735,4115190,0.020001,0.065668,2124-08-17 20:55:00,2124-08-17 21:33:00
31492,39995735,8549893,0.010001,0.024193,2124-08-17 21:33:00,2124-08-17 22:01:00
31472,39995735,3517354,0.020001,0.207373,2124-08-17 22:01:00,2124-08-18 00:01:00
31461,39995735,299691,0.010001,0.735309,2124-08-18 00:01:00,2124-08-18 14:12:00


## Python conversion of norepinephrine_dose.sql for MIMIC-IV

In [5]:
import pandas as pd
import numpy as np
from datetime import timedelta

def create_norepinephrine_dose(mimic_path, output_path, weightdurations=None):
    """
    Python conversion of norepinephrine_dose.sql for MIMIC-IV
    Creates a dataframe with norepinephrine dose and duration information

    Parameters:
    mimic_path (str): Path to MIMIC-IV data
    output_path (str): Path to save the output dataframe
    weightdurations (pd.DataFrame, optional): Weightdurations dataframe

    Returns:
    pd.DataFrame: Norepinephrine dose dataframe
    """
    print("Creating norepinephrine dose dataframe...")

    # load inputevents table (combined table in MIMIC-IV)
    print("Loading inputevents table...")
    try:
        inputevents = pd.read_csv(f"{mimic_path}/icu/inputevents.csv.gz",
                             usecols=['stay_id', 'starttime', 'endtime', 'itemid',
                                      'rate', 'amount', 'orderid', 'statusdescription'],
                             parse_dates=['starttime', 'endtime'])
        print(f"Successfully loaded inputevents. Shape: {inputevents.shape}, Type: {type(inputevents)}")
    except FileNotFoundError:
        print(f"Error: inputevents.csv.gz not found at {mimic_path}/icu/")
        return None # Exit the function if the file is not found
    except Exception as e:
        print(f"An error occurred while loading inputevents: {e}")
        return None # Exit the function if there's an error during loading

    # Rename stay_id to icustay_id for consistency with original SQL
    inputevents.rename(columns={'stay_id': 'icustay_id', 'orderid': 'linkorderid'}, inplace=True)

    # Load weightdurations if not provided
    if weightdurations is None:
        try:
            weightdurations = pd.read_csv(f"{output_path}/weightdurations.csv",
                                          parse_dates=['starttime', 'endtime'])
        except:
            print("weightdurations file not found. Weight-based calculations may be inaccurate.")
            weightdurations = None

    # Filter for norepinephrine
    # Note: MIMIC-IV itemids may differ from MIMIC-III
    # You'll need to update these itemids based on MIMIC-IV dictionary
    norepinephrine_itemid = 221906  # Update this with the correct MIMIC-IV itemid for norepinephrine

    # Check if inputevents is defined before filtering
    if 'inputevents' in locals() and isinstance(inputevents, pd.DataFrame):
        print("inputevents DataFrame is defined and ready for filtering.")
        # Filter for valid orders
        norepinephrine_mv = inputevents[
            (inputevents['itemid'] == norepinephrine_itemid) &
            (inputevents['statusdescription'] != 'Rewritten')
        ].copy()

        # If weightdurations is available, join to get weight-adjusted rates
        if weightdurations is not None:
            # This is a simplification of the SQL logic
            def find_weight(row):
                matching_weights = weightdurations[
                    (weightdurations['icustay_id'] == row['icustay_id']) &
                    (row['starttime'] >= weightdurations['starttime']) &
                    (row['starttime'] <= weightdurations['endtime'])
                ]
                if not matching_weights.empty:
                  return matching_weights.iloc[0]['weight']
                return 80.0  # Default weight if no match

            norepinephrine_mv['weight'] = norepinephrine_mv.apply(find_weight, axis=1)

        # Adjust rate based on weight where needed
        # This is a simplification of the SQL logic
            norepinephrine_mv['vaso_rate'] = norepinephrine_mv['rate']  # Already in mcgkgmin in MIMIC-IV
        else:
            norepinephrine_mv['vaso_rate'] = norepinephrine_mv['rate']

        # Group by icustay_id and linkorderid
        norepinephrine_mv_grouped = norepinephrine_mv.groupby(['icustay_id', 'linkorderid']).agg({
            'vaso_rate': 'max',
            'amount': 'sum',
            'starttime': 'min',
            'endtime': 'max'
        }).reset_index()

        norepinephrine_mv_grouped.rename(columns={
            'amount': 'vaso_amount'
        }, inplace=True)

        # Sort the final dataframe
        norepinephrine_dose = norepinephrine_mv_grouped.sort_values(['icustay_id', 'starttime'])

        # Save to CSV
        norepinephrine_dose.to_csv(f"{output_path}/norepinephrine_dose.csv", index=False)
        print(f"Norepinephrine dose saved to {output_path}/norepinephrine_dose.csv")

        return norepinephrine_dose
    else:
        print("inputevents DataFrame was not successfully created.")
        return None


# Call the function directly with the defined paths
print(f"Using mimic_path: {mimic_path}")
print(f"Using output_path: {output_path}")
create_norepinephrine_dose(mimic_path, output_path)

Using mimic_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0
Using output_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/
Creating norepinephrine dose dataframe...
Loading inputevents table...
Successfully loaded inputevents. Shape: (10953713, 8), Type: <class 'pandas.core.frame.DataFrame'>
inputevents DataFrame is defined and ready for filtering.
Norepinephrine dose saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data//norepinephrine_dose.csv


,icustay_id,linkorderid,vaso_rate,vaso_amount,starttime,endtime
0,30001446,146812,0.040011,0.267303,2186-04-12 09:57:00,2186-04-12 10:53:00
5,30001446,8528079,0.080021,1.069212,2186-04-12 10:53:00,2186-04-12 12:45:00
4,30001446,5020900,0.059994,0.501014,2186-04-12 12:45:00,2186-04-12 13:55:00
6,30001446,8711250,0.039981,0.643911,2186-04-12 13:55:00,2186-04-12 16:10:00
1,30001446,1476613,0.039981,0.014309,2186-04-12 16:10:00,2186-04-12 16:13:00
...,...,...,...,...,...,...
459798,39999230,9137908,0.159902,0.856274,2147-09-01 11:30:00,2147-09-01 13:00:00
459782,39999230,346061,0.100165,1.335000,2147-09-01 13:00:00,2147-09-01 16:44:00
459783,39999230,1323409,0.080032,0.361905,2147-09-01 16:44:00,2147-09-01 18:00:00
459787,39999230,3839646,0.060015,0.174973,2147-09-01 18:00:00,2147-09-01 18:49:00


## Python conversion of phenylephrine_dose.sql for MIMIC-IV

In [ ]:
# Python conversion of phenylephrine_dose.sql for MIMIC-IV
# Creates a dataframe with phenylephrine dose and duration information

import pandas as pd
import numpy as np
from datetime import timedelta

def create_phenylephrine_dose(mimic_path, output_path):
    """
    Python conversion of phenylephrine_dose.sql for MIMIC-IV
    Creates a dataframe with phenylephrine dose and duration information

    Parameters:
        mimic_path (str): Path to MIMIC-IV data
        output_path (str): Path to save the output dataframe

    Returns:
        pd.DataFrame: Phenylephrine dose dataframe
    """
    print("Creating phenylephrine dose dataframe...")

    # Load inputevents table (combined table in MIMIC-IV)
    print("Loading inputevents table...")
    inputevents = pd.read_csv(f"{mimic_path}/icu/inputevents.csv.gz",
                             usecols=['stay_id', 'starttime', 'endtime', 'itemid',
                                      'rate', 'amount', 'orderid', 'statusdescription'],
                             parse_dates=['starttime', 'endtime'])

    # Rename stay_id to icustay_id for consistency with original SQL
    inputevents.rename(columns={'stay_id': 'icustay_id', 'orderid': 'linkorderid'}, inplace=True)

    # Filter for phenylephrine
    # Note: MIMIC-IV itemids may differ from MIMIC-III
    # You'll need to update these itemids based on MIMIC-IV dictionary
    phenylephrine_itemid = 221749  # Update this with the correct MIMIC-IV itemid for phenylephrine

    # Filter for valid orders
    phenylephrine_mv = inputevents[
        (inputevents['itemid'] == phenylephrine_itemid) &
        (inputevents['statusdescription'] != 'Rewritten')
    ].copy()

    # Group by icustay_id and linkorderid
    phenylephrine_mv_grouped = phenylephrine_mv.groupby(['icustay_id', 'linkorderid']).agg({
        'rate': 'max',
        'amount': 'sum',
        'starttime': 'min',
        'endtime': 'max'
    }).reset_index()

    phenylephrine_mv_grouped.rename(columns={
        'rate': 'vaso_rate',
        'amount': 'vaso_amount'
    }, inplace=True)

    # Sort the final dataframe
    phenylephrine_dose = phenylephrine_mv_grouped.sort_values(['icustay_id', 'starttime'])

    # Save to CSV
    phenylephrine_dose.to_csv(f"{output_path}/phenylephrine_dose.csv", index=False)
    print(f"Phenylephrine dose saved to {output_path}/phenylephrine_dose.csv")

    return phenylephrine_dose

# Call the function directly with the defined paths
print(f"Using mimic_path: {mimic_path}")
print(f"Using output_path: {output_path}")
create_phenylephrine_dose(mimic_path, output_path)

Using mimic_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0
Using output_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/
Creating phenylephrine dose dataframe...
Loading inputevents table...
Phenylephrine dose saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data//phenylephrine_dose.csv


,icustay_id,linkorderid,vaso_rate,vaso_amount,starttime,endtime
3,30000646,8638918,0.500217,0.535032,2194-04-29 08:54:00,2194-04-29 09:08:00
0,30000646,1472834,0.799936,27.562898,2194-04-29 09:08:00,2194-04-29 16:39:00
2,30000646,7766485,0.599952,2.291815,2194-04-29 16:39:00,2194-04-29 17:29:00
1,30000646,4715229,0.500089,1.948546,2194-04-29 17:29:00,2194-04-29 18:20:00
4,30000646,9923344,0.299970,0.458355,2194-04-29 18:20:00,2194-04-29 18:40:00
...,...,...,...,...,...,...
209371,39996783,7250446,5.005536,36.580455,2126-06-29 06:22:00,2126-06-29 09:16:00
209374,39999552,5030737,0.200007,0.670823,2186-07-17 16:46:00,2186-07-17 17:38:00
209372,39999552,3219327,0.300075,1.064516,2186-07-17 19:30:00,2186-07-17 20:25:00
209375,39999552,6623711,0.400100,0.825806,2186-07-17 21:05:00,2186-07-17 21:37:00


## Python conversion of vasopressin_dose.sql for MIMIC-IV

In [6]:
# Python conversion of vasopressin_dose.sql for MIMIC-IV

import pandas as pd
import numpy as np
from datetime import timedelta

def create_vasopressin_dose(mimic_path, output_path, weightdurations=None):
    """
    Python conversion of vasopressin_dose.sql for MIMIC-IV
    Creates a dataframe with vasopressin dose and duration information

    Parameters:
        mimic_path (str): Path to MIMIC-IV data
        output_path (str): Path to save the output dataframe
        weightdurations (pd.DataFrame, optional): Weightdurations dataframe

    Returns:
        pd.DataFrame: Vasopressin dose dataframe
    """
    print("Creating vasopressin dose dataframe...")

    # Load inputevents table (combined table in MIMIC-IV)
    print("Loading inputevents table...")
    try:
        inputevents = pd.read_csv(f"{mimic_path}/icu/inputevents.csv.gz",
                             usecols=['stay_id', 'starttime', 'endtime', 'itemid',
                                      'rate', 'amount', 'orderid', 'statusdescription'],
                             parse_dates=['starttime', 'endtime'])
        print(f"Successfully loaded inputevents. Shape: {inputevents.shape}, Type: {type(inputevents)}")
    except FileNotFoundError:
        print(f"Error: inputevents.csv.gz not found at {mimic_path}/icu/")
        return None # Exit the function if the file is not found
    except Exception as e:
        print(f"An error occurred while loading inputevents: {e}")
        return None # Exit the function if there's an error during loading


    # Rename stay_id to icustay_id for consistency with original SQL
    inputevents.rename(
        columns={'stay_id': 'icustay_id', 'orderid': 'linkorderid'},
        inplace=True
    )

    # Load weightdurations if not provided
    if weightdurations is None:
        try:
            weightdurations = pd.read_csv(
                f"{output_path}/weightdurations.csv",
                parse_dates=['starttime', 'endtime']
            )
        except:
          print("Weightdurations file not found. Weight-based calculations may be inaccurate.")
          weightdurations = None

    # Filter for vasopressin
    # Note: MIMIC-IV itemids may differ from MIMIC-III
    # You'll need to update these itemids based on MIMIC-IV dictionary
    vasopressin_itemid = 222315  # Update this with the correct MIMIC-IV itemid for vasopressin

    # Check if inputevents is defined before filtering
    if 'inputevents' in locals() and isinstance(inputevents, pd.DataFrame):
        print("inputevents DataFrame is defined and ready for filtering.")
        # Filter for valid orders
        vasopressin_mv = inputevents[
            (inputevents['itemid'] == vasopressin_itemid) &
            (inputevents['statusdescription'] != 'Rewritten')
        ].copy()

        # If weightdurations is available, join to get weight-adjusted rates
        if weightdurations is not None:
            # For each infusion, find the appropriate weight
            # This is a simplification of the SQL logic
            def find_weight(row):
                matching_weights = weightdurations[
                    (weightdurations['icustay_id'] == row['icustay_id']) &
                    (row['starttime'] >= weightdurations['starttime']) &
                    (row['starttime'] <= weightdurations['endtime'])
                ]
                if not matching_weights.empty:
                    return matching_weights.iloc[0]['weight']
                return 80.0  # Default weight if no match

            vasopressin_mv['weight'] = vasopressin_mv.apply(find_weight, axis=1)

            # Adjust rate based on weight where needed
            # This is a simplification of the SQL logic
            vasopressin_mv['vaso_rate'] = vasopressin_mv['rate']  # Already in mcgkgmin in MIMIC-IV
        else:
            vasopressin_mv['vaso_rate'] = vasopressin_mv['rate']

        # Group by icustay_id and linkorderid
        vasopressin_mv_grouped = vasopressin_mv.groupby(['icustay_id', 'linkorderid']).agg({
            'vaso_rate': 'max',
            'amount': 'sum',
            'starttime': 'min',
            'endtime': 'max'
        }).reset_index()

        vasopressin_mv_grouped.rename(columns={
            'amount': 'vaso_amount'
        }, inplace=True)

        # Sort the final dataframe
        vasopressin_dose = vasopressin_mv_grouped.sort_values(['icustay_id', 'starttime'])

        # Save to CSV
        vasopressin_dose.to_csv(f"{output_path}/vasopressin_dose.csv", index=False)
        print(f"Vasopressin dose saved to {output_path}/vasopressin_dose.csv")

        return vasopressin_dose
    else:
        print("inputevents DataFrame was not successfully created.")
        return None


# Call the function directly with the defined paths
print(f"Using mimic_path: {mimic_path}")
print(f"Using output_path: {output_path}")
create_vasopressin_dose(mimic_path, output_path)

Using mimic_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0
Using output_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/
Creating vasopressin dose dataframe...
Loading inputevents table...
Successfully loaded inputevents. Shape: (10953713, 8), Type: <class 'pandas.core.frame.DataFrame'>
inputevents DataFrame is defined and ready for filtering.
Vasopressin dose saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data//vasopressin_dose.csv


,icustay_id,linkorderid,vaso_rate,vaso_amount,starttime,endtime
3,30003749,9966246,3.601810,39.800000,2180-06-06 16:30:00,2180-06-07 03:33:00
0,30003749,2893596,3.601804,39.920001,2180-06-07 03:33:00,2180-06-07 14:38:00
1,30003749,6786994,3.603604,7.027027,2180-06-07 14:38:00,2180-06-07 16:35:00
2,30003749,9508139,2.403862,26.442481,2180-06-07 16:35:00,2180-06-08 03:35:00
4,30004391,6726088,1.200000,9.160000,2153-09-05 17:32:00,2153-09-06 01:10:00
...,...,...,...,...,...,...
37162,39998012,9645041,1.199998,4.479994,2133-02-05 23:00:00,2133-02-06 02:44:00
37159,39998012,6781646,2.438503,7.600001,2133-02-06 02:44:00,2133-02-06 05:51:00
37155,39998012,751388,2.399997,6.199992,2133-02-06 05:51:00,2133-02-06 08:26:00
37157,39998012,3849408,1.200000,11.220001,2133-02-06 08:26:00,2133-02-06 17:47:00


## Python conversion of weight_duration.sql for MIMIC-IV

In [2]:
import pandas as pd
import numpy as np
from datetime import timedelta

def create_weight_durations(mimic_path, output_path):
    """
    Python conversion of weight-durations.sql for MIMIC-IV
    Creates a dataframe with weights for ICU patients with start/stop times

    Parameters:
        mimic_path (str): Path to MIMIC-IV data

        output_path (str): Path to save the output dataframe

    Returns:
        pd.DataFrame: Weight durations dataframe
    """

    print("Creating weight durations dataframe...")

    # Load necessary tables
    print("Loading chartevents and icustays tables...")

    # In MIMIC-IV, chartevents structure is similar but may have different itemids
    chartevents = pd.read_csv(f"{mimic_path}/icu/chartevents.csv.gz",
                             usecols=['stay_id', 'charttime', 'itemid', 'valuenum'],
                             parse_dates=['charttime'])

    # Filter for weight measurements
    # Note: MIMIC-IV itemids may differ from MIMIC-III
    # You'll need to update these itemids based on MIMIC-IV dictionary
    weight_itemids = [
        762, 226512,    # Admit Weight
        763, 226531     # Daily Weight
    ]

    # Filter chartevents for weight measurements
    wt_stg = chartevents[
        (chartevents['itemid'].isin(weight_itemids)) &
        (chartevents['valuenum'].notna()) &
        (chartevents['valuenum'] != 0)
    ].copy()

    # Add weight_type column
    wt_stg['weight_type'] = wt_stg['itemid'].apply(
        lambda x: 'admit' if x in [762, 226512] else 'daily'
    )

    # Select relevant columns
    wt_stg = wt_stg[['stay_id', 'charttime', 'weight_type', 'valuenum']]
    wt_stg.rename(columns={'valuenum': 'weight', 'stay_id': 'icustay_id'}, inplace=True)

    # Load icustays table
    icustays = pd.read_csv(f"{mimic_path}/icu/icustays.csv.gz",
                           usecols=['stay_id', 'intime', 'outtime'],
                           parse_dates=['intime', 'outtime'])
    icustays.rename(columns={'stay_id': 'icustay_id'}, inplace=True)

    # Assign row numbers within groups
    wt_stg['rn'] = wt_stg.groupby(['icustay_id', 'weight_type']).cumcount() + 1

    # Merge with icustays to get intime and outtime
    wt_stg2 = pd.merge(wt_stg, icustays, on='icustay_id', how='inner')

    # Set starttime based on weight_type
    wt_stg2['starttime'] = wt_stg2.apply(
        lambda row: row['intime'] + timedelta(hours=2) if row['weight_type'] == 'admit' and row['rn'] == 1
        else row['charttime'],
        axis=1
    )

    # Filter out rows where weight_type is 'admit' and rn is 1
    wt_stg2 = wt_stg2[~((wt_stg2['weight_type'] == 'admit') & (wt_stg2['rn'] == 1))]

    # Calculate endtime for each weight entry
    wt_stg2 = wt_stg2.sort_values(['icustay_id', 'starttime'])
    wt_stg3 = wt_stg2.groupby('icustay_id')['starttime'].shift(-1)
    wt_stg2['endtime'] = wt_stg3
    wt_stg2['endtime'] = wt_stg2['endtime'].fillna(wt_stg2['outtime'] + timedelta(hours=2))

    # Select only relevant columns
    wt_stg2 = wt_stg2[['icustay_id', 'starttime', 'endtime', 'weight']]

    # Create weight table for matching with event intervals
    wt1 = pd.merge(icustays[['icustay_id', 'intime', 'outtime']],
                   wt_stg2,
                   on='icustay_id',
                   how='left')

    # Calculate endtime for wt1 intervals
    wt1_grouped = wt1.sort_values(['icustay_id', 'starttime'])
    wt1_grouped['next_starttime'] = wt1_grouped.groupby('icustay_id')['starttime'].shift(-1)

    # Assign endtime
    wt1_grouped['endtime'] = wt1_grouped.apply(
        lambda row: row['next_starttime'] if pd.notna(row['next_starttime'])
                    else row['outtime'] + timedelta(hours=2), axis=1)
    wt1 = wt1_grouped

    # Fix gaps at start of stays: find first weight for each icustay_id
    first_weights = wt1_grouped.sort_values(['icustay_id', 'starttime']) \
        .groupby('icustay_id').first().reset_index()
    first_weights = first_weights[['icustay_id', 'starttime', 'weight']]

    # Align icustays where intime < first weight starttime
    wt_fix = pd.merge(icustays[['icustay_id', 'intime']],
                      first_weights,
                      on='icustay_id',
                      how='inner')
    wt_fix = wt_fix[wt_fix['intime'] < wt_fix['starttime']].copy()

    # Create new rows for the gap period
    if not wt_fix.empty:
        wt_fix['endtime'] = wt_fix['starttime']
        wt_fix['starttime'] = wt_fix['intime'] - timedelta(hours=2)
        wt_fix = wt_fix[['icustay_id', 'starttime', 'endtime', 'weight']]
        # Combine with wt1
        wt2 = pd.concat([wt1_grouped[['icustay_id', 'starttime', 'endtime', 'weight']], wt_fix], ignore_index=True)
    else:
        wt2 = wt1_grouped[['icustay_id', 'starttime', 'endtime', 'weight']]
    # Sort the final dataframe
    weightdurations = wt2.sort_values(['icustay_id', 'starttime', 'endtime'])
    # Save to CSV
    weightdurations.to_csv(f"{output_path}/weightdurations.csv", index=False)
    print(f"Weight durations saved to {output_path}/weightdurations.csv")
    return weightdurations

# Call the function directly with the defined paths
print(f"Using mimic_path: {mimic_path}")
print(f"Using output_path: {output_path}")
create_weight_durations(mimic_path, output_path)

Using mimic_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0
Using output_path: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/
Creating weight durations dataframe...
Loading chartevents and icustays tables...
Weight durations saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data//weightdurations.csv


,icustay_id,starttime,endtime,weight
0,30000153,NaT,2174-10-01 05:26:10,NaN
1,30000213,NaT,2162-06-22 22:52:48,NaN
225094,30000484,2136-01-14 15:23:32,2136-01-14 18:46:00,150.7
2,30000484,2136-01-14 18:46:00,2136-01-15 10:14:00,150.7
3,30000484,2136-01-15 10:14:00,2136-01-15 14:58:00,150.7
...,...,...,...,...
203565,39999562,2129-01-24 15:24:09,2129-01-29 11:42:00,136.4
164942,39999562,2129-01-29 11:42:00,2129-01-29 19:18:49,136.4
220813,39999810,2115-11-30 22:37:00,2115-12-01 06:38:00,158.4
164943,39999810,2115-12-01 06:38:00,2115-12-05 20:27:57,158.4
